# Dispense Pipeline — End-to-End Test

Tests the full path: **Backend API → pending_command → ESP32 motor → command_result**

### Prerequisites
1. `uvicorn backend:app --host 0.0.0.0 --port 8000` is running
2. ESP32 is powered on and connected to the same WiFi
3. Update `BASE_URL` below if needed

In [1]:
import requests, time, json

BASE_URL = "http://192.168.197.86:8000"  # ← update to your backend IP

def api(method, path, body=None):
    url = BASE_URL + path
    r = requests.request(method, url, json=body, timeout=5)
    print(f"{method} {path} → {r.status_code}")
    data = r.json()
    print(json.dumps(data, indent=2))
    return data

print("Helper loaded. Base URL:", BASE_URL)

Helper loaded. Base URL: http://192.168.197.86:8000


## 1. Check current state & reset

In [2]:
# Get current system state
state = api("GET", "/state")
print("\n→ demo_state:", state["demo_state"])
print("→ pending_command:", state["pending_command"])
print("→ dispense_status:", state["dispense_status"])

GET /state → 200
{
  "demo_state": "idle",
  "candidate_ingredients": [],
  "confirmed_ingredients": [],
  "recipe": null,
  "current_step_index": -1,
  "containers": [
    "salt",
    "black pepper",
    "garlic powder"
  ],
  "container_levels": [
    100,
    100,
    100
  ],
  "weight": 0.0,
  "dispense_status": "idle",
  "manual_override": false,
  "resume_state": "idle",
  "last_error": null,
  "pending_command": null,
  "planner_mode": "llm",
  "last_planner_status": null,
  "last_dispense_actual_grams": null
}

→ demo_state: idle
→ pending_command: None
→ dispense_status: idle


In [3]:
# Reset to idle
api("POST", "/reset")

POST /reset → 200
{
  "ok": true,
  "demo_state": "idle"
}


{'ok': True, 'demo_state': 'idle'}

## 2. Confirm ingredients & generate recipe

In [4]:
# Confirm some test ingredients
api("POST", "/ingredients_confirmed", {"ingredients": ["chicken", "rice"]})

POST /ingredients_confirmed → 200
{
  "ok": true,
  "demo_state": "ingredients_confirmed",
  "confirmed_ingredients": [
    "chicken",
    "rice"
  ]
}


{'ok': True,
 'demo_state': 'ingredients_confirmed',
 'confirmed_ingredients': ['chicken', 'rice']}

In [5]:
# Generate recipe (uses mock recipe generator)
recipe_resp = api("POST", "/generate_recipe", {})

print("\n── Recipe Steps ──")
for step in recipe_resp["recipe"]["steps"]:
    tag = "🧂" if step["type"] == "dispense" else "🍳"
    print(f"  {tag} Step {step['index']}: {step['instruction']}")
    if step["dispense"]:
        print(f"     → dispense: {step['dispense']['spice']} {step['dispense']['grams']}g")

POST /generate_recipe → 200
{
  "ok": true,
  "demo_state": "recipe_ready",
  "planner_mode": "llm",
  "planner_status": "openai_sdk_or_api_key_missing_fallback_to_mock",
  "recipe": {
    "name": "AI Chef's Special",
    "description": "A recipe generated from: chicken, rice.",
    "steps": [
      {
        "index": 0,
        "instruction": "Prepare your ingredients: chicken, rice.",
        "type": "cook",
        "dispense": null
      },
      {
        "index": 1,
        "instruction": "Dispense salt into the bowl.",
        "type": "dispense",
        "dispense": {
          "spice": "salt",
          "grams": 3.0
        }
      },
      {
        "index": 2,
        "instruction": "Dispense black pepper into the bowl.",
        "type": "dispense",
        "dispense": {
          "spice": "black pepper",
          "grams": 1.5
        }
      },
      {
        "index": 3,
        "instruction": "Cook on medium heat for 8 minutes.",
        "type": "cook",
        "dispense":

## 3. Advance to a dispense step

In [6]:
# Step 0 is a 'cook' step, advance to step 1 (salt dispense)
resp = api("POST", "/advance_step")
print("\n→ Now on step:", resp["step"]["index"], "-", resp["step"]["instruction"])
print("→ demo_state:", resp["demo_state"])

POST /advance_step → 200
{
  "ok": true,
  "demo_state": "dispensing_step",
  "step": {
    "index": 1,
    "instruction": "Dispense salt into the bowl.",
    "type": "dispense",
    "dispense": {
      "spice": "salt",
      "grams": 3.0
    }
  }
}

→ Now on step: 1 - Dispense salt into the bowl.
→ demo_state: dispensing_step


## 4. Trigger dispense — this sends the command to ESP32

In [ ]:
# Dispense! This writes pending_command which ESP32 picks up via GET /pending_command
dispense_resp = api("POST", "/dispense_step")

print("\n→ Command sent to ESP32:")
cmd = dispense_resp["command"]
print(f"  command_id:     {cmd['command_id']}")
print(f"  spice:          {cmd['spice']}")
print(f"  container_index:{cmd['container_index']}")
print(f"  target_grams:   {cmd['target_grams']}")

## 5. Verify ESP32 picks up the command

In [ ]:
# Check what ESP32 would see when it polls /pending_command
api("GET", "/pending_command")

## 6. Poll until ESP32 reports back (or timeout)

In [ ]:
# Wait for ESP32 to finish dispensing and report /command_result
# The pending_command will be cleared when it does.

TIMEOUT = 20  # seconds
start = time.time()

print("Waiting for ESP32 to complete dispense...")
while time.time() - start < TIMEOUT:
    state = requests.get(BASE_URL + "/state", timeout=5).json()
    status = state["dispense_status"]
    pending = state["pending_command"]
    weight = state["weight"]
    elapsed = time.time() - start
    
    print(f"  [{elapsed:5.1f}s] dispense_status={status}  weight={weight:.2f}g  pending={'yes' if pending else 'no'}")
    
    if status in ("done", "error"):
        print(f"\n✅ Dispense finished: {status}")
        print(f"   Final weight: {weight:.2f}g")
        if state["last_error"]:
            print(f"   Error: {state['last_error']}")
        break
    time.sleep(1)
else:
    print(f"\n⏰ Timed out after {TIMEOUT}s — ESP32 may not be connected.")
    print("   Check: Is ESP32 on? Is it on the same WiFi? Is BACKEND IP correct in main.cpp?")

## 7. (Optional) Simulate ESP32 response manually
If the ESP32 isn't connected, you can fake the response to test the backend side:

In [ ]:
# Only run this if ESP32 is NOT connected and you want to test backend logic
state = requests.get(BASE_URL + "/state", timeout=5).json()
pending = state.get("pending_command")

if pending and pending.get("command_id"):
    fake_result = {
        "command_id": pending["command_id"],
        "status": "done",
        "actual_grams": pending["target_grams"],
        "weight": 3.0
    }
    print("Sending fake command_result:", json.dumps(fake_result, indent=2))
    api("POST", "/command_result", fake_result)
else:
    print("No pending command — either ESP32 already handled it or no dispense was triggered.")

## 8. Final state check

In [ ]:
state = api("GET", "/state")
print("\n── Summary ──")
print(f"  demo_state:      {state['demo_state']}")
print(f"  dispense_status: {state['dispense_status']}")
print(f"  weight:          {state['weight']:.2f}g")
print(f"  pending_command: {state['pending_command']}")
print(f"  current_step:    {state['current_step_index']}")
print(f"  last_error:      {state['last_error']}")

## 9. Manual dispense test (bypass recipe)
Use manual override to dispense directly without going through the recipe flow:

In [ ]:
# Enable manual override
api("POST", "/manual_override", {"active": True})

# Dispense 2g of salt directly
api("POST", "/manual_dispense", {"spice": "salt", "grams": 2.0})

In [ ]:
# Disable manual override when done
api("POST", "/manual_override", {"active": False})